# 02 Cleaning

Use this notebook to clean the raw data, document your transformations, and export a processed file to `data/processed/`.

In [7]:
import pandas as pd
import os

raw_data_path = '../data/raw/'
processed_data_path = '../data/processed/'

os.makedirs(processed_data_path, exist_ok=True)

# Load data
account_statuses = pd.read_csv(os.path.join(raw_data_path, 'account_statuses.csv'))
account_types = pd.read_csv(os.path.join(raw_data_path, 'account_types.csv'))
accounts = pd.read_csv(os.path.join(raw_data_path, 'accounts.csv'))
addresses = pd.read_csv(os.path.join(raw_data_path, 'addresses.csv'))
branches = pd.read_csv(os.path.join(raw_data_path, 'branches.csv'))
customer_types = pd.read_csv(os.path.join(raw_data_path, 'customer_types.csv'))
customers = pd.read_csv(os.path.join(raw_data_path, 'customers.csv'))
loan_statuses = pd.read_csv(os.path.join(raw_data_path, 'loan_statuses.csv'))
loans = pd.read_csv(os.path.join(raw_data_path, 'loans.csv'))
transaction_types = pd.read_csv(os.path.join(raw_data_path, 'transaction_types.csv'))
transactions = pd.read_csv(os.path.join(raw_data_path, 'transactions.csv'))

## 1. Create dim_customers_accounts
We join customers with their addresses, types, and then link them to their accounts.

In [8]:
# Join customers with types and addresses
customers_full = customers.merge(customer_types, on='CustomerTypeID', how='left', suffixes=('', '_Type'))
customers_full = customers_full.merge(addresses, on='AddressID', how='left')

# Join accounts with types and statuses
accounts_full = accounts.merge(account_types, on='AccountTypeID', how='left', suffixes=('', '_Type'))
accounts_full = accounts_full.merge(account_statuses, on='AccountStatusID', how='left', suffixes=('', '_Status'))

# Consolidate Customer and Account information
dim_customers_accounts = accounts_full.merge(customers_full, on='CustomerID', how='left', suffixes=('_Account', '_Customer'))

print(f"dim_customers_accounts shape: {dim_customers_accounts.shape}")
dim_customers_accounts.head()

dim_customers_accounts shape: (1758, 17)


,AccountID,CustomerID,AccountTypeID,AccountStatusID,Balance,OpeningDate,TypeName_Account,StatusName,FirstName,LastName,DateOfBirth,AddressID,CustomerTypeID,TypeName_Customer,Street,City,Country
0,200094,10123,3,1,48348.54,2018-06-12 00:00:00.000000,Payroll,Active,Cecille,Spencer,1993-12-05 00:00:00.000000,499,2,Small Business,Girard,Menasha,United States
1,201108,10077,3,1,35001.41,2019-10-30 00:00:00.000000,Payroll,Active,Dario,Campbell,1964-04-16 00:00:00.000000,12,2,Small Business,Judson,Bellwood,United States
2,201453,10321,3,2,57081.03,2020-05-24 00:00:00.000000,Payroll,Inactive,Mack,Goodwin,1989-09-18 00:00:00.000000,246,3,Large Enterprise,Michigan,Edina,United States
3,200581,10871,5,1,63164.33,2021-01-27 00:00:00.000000,Youth,Active,Tiffiny,Vinson,1964-05-09 00:00:00.000000,362,2,Small Business,Starview,Manhattan,United States
4,200003,10765,1,1,58739.64,2018-09-12 00:00:00.000000,Checking,Active,Junior,Witt,1992-07-09 00:00:00.000000,870,1,Individual,Murray,Marshalltown,United States


## 2. Create fact_transactions

In [9]:
# Join branches with their addresses
branches_full = branches.merge(addresses, on='AddressID', how='left', suffixes=('', '_Branch'))

# Join transactions with types and branches
fact_transactions = transactions.merge(transaction_types, on='TransactionTypeID', how='left', suffixes=('', '_Type'))
fact_transactions = fact_transactions.merge(branches_full, on='BranchID', how='left', suffixes=('', '_Branch'))

print(f"fact_transactions shape: {fact_transactions.shape}")

fact_transactions shape: (50995, 14)


## 3. Create fact_loans

In [10]:
# Join loans with statuses
fact_loans = loans.merge(loan_statuses, on='LoanStatusID', how='left', suffixes=('', '_Status'))
print(f"fact_loans shape: {fact_loans.shape}")

fact_loans shape: (333, 8)


## Save the consolidated tables

In [11]:
dim_customers_accounts.to_csv(os.path.join(processed_data_path, 'dim_customers_accounts.csv'), index=False)
fact_transactions.to_csv(os.path.join(processed_data_path, 'fact_transactions.csv'), index=False)
fact_loans.to_csv(os.path.join(processed_data_path, 'fact_loans.csv'), index=False)
print("Consolidated tables saved to data/processed/")

Consolidated tables saved to data/processed/
